## 03. 세입예산편성현황 데이터 분석

groupby를 이용한 집계(SQL의 GROUP BY 와 동일)
- 데이터프레임`.groupby('컬럼명')`
- 데이터프레임`.groupby('그룹화기준컬럼')['그룹화 대상 컬럼].집계함수`
- 데이터프레임.`sort_values('정렬기준컬럽',ascending = True | False)` 

데이터 차이 분석(정부안금액 - 국회확정금액 차이 col 추가)
- 데이터프레임`['신규컬럼명'] = 컬럼값`
- `세입예산편성_DF['차이금액'] = 세입예산편성_DF.정부안금액 - 세입예산편성_DF.국회확정금액`

IF 차이금액>0, 수입목명의 빈도 분석
- `pd.DataFrame(세입예산편성_DF[세입예산편성_DF.차이금액 < 0].수입목명.value_counts())`

세입 과목 분류별 분석
![세입 과목 분류별 분석 이미지](Practice%20Data/세입분류명.png)
- `apply()` 함수를 이용한 세입 과목 분류표 적용 및 신규 컬럼 생성

iloc를 이용한 컬럼 행출력 순서 지정(특정 행 추출, 행과열 순서 변경 가능)
- 데이터프레임`.iloc[[0,2,3,1],:]` (행 순서 변경)
- 데이터프레임`.iloc[행인덱스, 열인덱스]`

axis=0 vs axis=1
- `axis=0` : 열단위, 모든 행의 평균이나 열별 합계 구할 때
- `axis=1` : 행단위, 여러 열을 조합해서 새로운 값 만들 때

## 

In [2]:
!pip3 install pandas
!pip3 install koreanize_matplotlib
!pip3 install seaborn

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib

# 과학적 표기법 해제
pd.options.display.float_format = '{:.0f}'.format

세입예산편성_DF = pd.read_csv("Practice Data/세입_예산편성현황_2024.csv")
세출예산편성_DF = pd.read_csv("Practice Data/세출_예산편성현황_2024.csv")


# 컬럼명 변경
세입예산편성_DF.rename(columns = {'정부안금액(천원)' : '정부안금액', '국회확정금액(천원)':'국회확정금액'}, inplace=True)

# 컴마 제거, 문자열 데이터 숫자 변환(정부안금액/국회확정금액)
세입예산편성_DF.정부안금액 = 세입예산편성_DF.정부안금액.str.replace(',', '')
세입예산편성_DF.국회확정금액 = 세입예산편성_DF.국회확정금액.str.replace(',', '')

세입예산편성_DF.정부안금액 = pd.to_numeric(세입예산편성_DF.정부안금액)
세입예산편성_DF.국회확정금액 = pd.to_numeric(세입예산편성_DF.국회확정금액)


# 소관명 빈도 분석 및 빈도 비율 분석
pd.DataFrame(세입예산편성_DF.소관명.value_counts())

# 위에서 과학적 표기법으로 정수형태로 반환하였기에, 특정 코드에서만 임시로 보이게 설정해줘야함.
# 이 블록 안에서만 소수점 4자리로 출력함
with pd.option_context('display.float_format','{:.4}'.format):  
    pd.DataFrame(세입예산편성_DF.소관명.value_counts(normalize=True))

세입예산편성_DF

,No.,회계연도,소관명,회계명,계정명,수입관명,수입항명,수입목명,수입구분명,정부안금액,국회확정금액
0,1,2024,감사원,일반회계,NaN,경상이전수입,기타경상이전수입,기타경상이전수입,일반수입,30000,30000
1,2,2024,감사원,일반회계,NaN,수입대체경비수입,잡수입,기타잡수입,일반수입,600000,600000
2,3,2024,감사원,일반회계,NaN,재산수입,관유물대여료,건물대여료,일반수입,10000,10000
3,4,2024,감사원,일반회계,NaN,재산수입,관유물대여료,토지대여료,일반수입,1000,1000
4,5,2024,감사원,일반회계,NaN,재화및용역판매수입,잡수입,기타잡수입,일반수입,104000,104000
...,...,...,...,...,...,...,...,...,...,...,...
1748,1749,2024,환경부,환경개선특별회계,NaN,재산수입,기타이자수입및재산수입,기타재산이자외수입,일반수입,927000,927000
1749,1750,2024,환경부,환경개선특별회계,NaN,재산수입,기타이자수입및재산수입,지방자치단체이자수입,일반수입,1852000,1852000
1750,1751,2024,환경부,환경개선특별회계,NaN,재화및용역판매수입,면허료및수수료,면허료및수수료,일반수입,1242000,1242000
1751,1752,2024,환경부,환경개선특별회계,NaN,재화및용역판매수입,잡수입,기타잡수입,일반수입,3038000,3038000


In [ ]:
# 소관명별 정부안금액 합계
소관별_정부안금액 = pd.DataFrame(세입예산편성_DF.groupby('소관명')['정부안금액'].sum())
소관별_정부안금액.sort_values('정부안금액', ascending=False)
 
# 소관명별 국회확정금액 합계
소관별_국회확정금액 = pd.DataFrame(세입예산편성_DF.groupby('소관명')['국회확정금액'].sum())
소관별_국회확정금액.sort_values('국회확정금액', ascending=False)

       No.  회계연도  소관명       회계명  계정명       수입관명         수입항명        수입목명  \
0        1  2024  감사원      일반회계  NaN     경상이전수입     기타경상이전수입    기타경상이전수입   
1        2  2024  감사원      일반회계  NaN   수입대체경비수입          잡수입       기타잡수입   
2        3  2024  감사원      일반회계  NaN       재산수입       관유물대여료       건물대여료   
3        4  2024  감사원      일반회계  NaN       재산수입       관유물대여료       토지대여료   
4        5  2024  감사원      일반회계  NaN  재화및용역판매수입          잡수입       기타잡수입   
...    ...   ...  ...       ...  ...        ...          ...         ...   
1748  1749  2024  환경부  환경개선특별회계  NaN       재산수입  기타이자수입및재산수입   기타재산이자외수입   
1749  1750  2024  환경부  환경개선특별회계  NaN       재산수입  기타이자수입및재산수입  지방자치단체이자수입   
1750  1751  2024  환경부  환경개선특별회계  NaN  재화및용역판매수입      면허료및수수료     면허료및수수료   
1751  1752  2024  환경부  환경개선특별회계  NaN  재화및용역판매수입          잡수입       기타잡수입   
1752  1753  2024  환경부  환경개선특별회계  NaN  정부내부수입및기타          전입금     일반회계전입금   

              수입구분명       정부안금액      국회확정금액  
0              일반수입       30000       300

In [ ]:
# 정부안금액 및 국회확정금액 차이 분석 + new column 생성
세입예산편성_DF['차이금액'] = 세입예산편성_DF.정부안금액 - 세입예산편성_DF.국회확정금액
print(세입예산편성_DF.head())

# 수입목명별 세입 축소 빈도 분석
print(pd.DataFrame(세입예산편성_DF[세입예산편성_DF.차이금액 > 0].수입목명.value_counts()))
print(pd.DataFrame(세입예산편성_DF[세입예산편성_DF.차이금액 < 0].수입목명.value_counts()))

   No.  회계연도  소관명   회계명  계정명       수입관명      수입항명      수입목명 수입구분명   정부안금액  \
0    1  2024  감사원  일반회계  NaN     경상이전수입  기타경상이전수입  기타경상이전수입  일반수입   30000   
1    2  2024  감사원  일반회계  NaN   수입대체경비수입       잡수입     기타잡수입  일반수입  600000   
2    3  2024  감사원  일반회계  NaN       재산수입    관유물대여료     건물대여료  일반수입   10000   
3    4  2024  감사원  일반회계  NaN       재산수입    관유물대여료     토지대여료  일반수입    1000   
4    5  2024  감사원  일반회계  NaN  재화및용역판매수입       잡수입     기타잡수입  일반수입  104000   

   국회확정금액  차이금액  
0   30000     0  
1  600000     0  
2   10000     0  
3    1000     0  
4  104000     0  
            count
수입목명             
일반회계전입금         6
기금예수금           5
기금예탁이자수입        2
기금전입금           2
계정간전입금          2
일반회계예탁이자수입      1
국공채발행수입         1
기타재산이자외수입       1
신고분             1
원천분             1
입장료수입           1
토지대여료           1
건물대여료           1
면허료및수수료         1
기타민간차입금         1
              count
수입목명               
일반회계전입금          16
기타경상이전수입          6
기금전입금             3
전년도세계잉여금          3
기금예

In [21]:
# 소관명별 정부안금액 및 국회확정금액 차이 분석
소관별_차이금액 = pd.DataFrame(세입예산편성_DF.groupby('소관명')['차이금액'].sum())

#차이금액별로 상위10개 출력
print(pd.DataFrame(소관별_차이금액.sort_values('차이금액',ascending=True)[:10]))
print(pd.DataFrame(소관별_차이금액.sort_values('차이금액',ascending=False)[:5]))


              차이금액
소관명               
중소벤처기업부 -345955000
국토교통부   -240195000
기획재정부   -237738360
교육부     -184964000
질병관리청   -172620000
조달청      -78188000
해양수산부    -30560000
환경부      -17062000
문화체육관광부   -8062000
농림축산식품부   -6023000
                 차이금액
소관명                  
산업통상자원부      29291000
법무부          28062000
행정중심복합도시건설청   7228000
국방부           4411000
국가보훈부         3704000


In [38]:
# 세입과목 분류표(한국재정정보원, 2024 회계기금운용기준)

국세수입 = ['내국세', '관세','교통·에너지·환경세','교육세', '농어촌특별세', '종합부동산세']
자체수입 = ['기업특별회계영업수입', '사회보장기여금','재산수입','경상이전수입','재화및용역판매수입',
'수입대체경비수입','관유물매각대','융자및전대차관원금회수']
정부내부수입및기타 = ['정부내부수입및기타','전년도이월금','세계잉여금','차입금및여유자금회수']
여유자금회수 = ['유가증권매각대','정부예금회수']

def 세입과목분류(row):
    if row.수입관명 in 국세수입:
        return '국세수입'
    elif row.수입관명 in 자체수입:
        return '자체수입'
    elif row.수입관명 in 정부내부수입및기타:
        if row.수입목명 == '기타민간예수금':
            return '여유자금회수'
        elif row.수입목명 in 여유자금회수:
            return '여유자금회수'
        else:
            return '정부내부수입및기타'
    

In [39]:
# 세입과목 분류표 적용
세입예산편성_DF['세입과목분류'] = 세입예산편성_DF.apply(세입과목분류,axis=1)
세입예산편성_DF.sample(10) # 랜덤출력

,No.,회계연도,소관명,회계명,계정명,수입관명,수입항명,수입목명,수입구분명,정부안금액,국회확정금액,차이금액,세입과목분류
237,238,2024,과학기술정보통신부,정보통신진흥기금,NaN,경상이전수입,기타경상이전수입,법정부담금,일반수입,503337000,503337000,0,자체수입
379,380,2024,국방부,국방·군사시설이전특별회계,NaN,관유물매각대,고정자산매각대,건물매각대,일반수입,3521000,3521000,0,자체수입
828,829,2024,농림축산식품부,농지관리기금,NaN,관유물매각대,재고자산매각대및유동자산,재고자산매각대,일반수입,23499000,23499000,0,자체수입
1571,1572,2024,해양경찰청,일반회계,NaN,재산수입,관유물대여료,기타관유물대여료,일반수입,5000,5000,0,자체수입
1689,1690,2024,환경부,농어촌구조개선특별회계,농어촌특별세사업계정,경상이전수입,기타경상이전수입,기타경상이전수입,일반수입,500000,500000,0,자체수입
1190,1191,2024,보건복지부,국민연금기금,NaN,차입금및여유자금회수,유가증권매각대,공채매각대,보전수입,200000000,200000000,0,정부내부수입및기타
796,797,2024,농림축산식품부,농어업재해재보험기금,NaN,경상이전수입,기타경상이전수입,기타경상이전수입,일반수입,23000,23000,0,자체수입
544,545,2024,국토교통부,주택도시기금,주택계정,정부내부수입및기타,예수금,기타민간예수금,보전수입,21750000000,21750000000,0,여유자금회수
1274,1275,2024,산업통상자원부,방사성폐기물관리기금,NaN,차입금및여유자금회수,정부예금회수,통화금융기관예치금회수,보전수입,323263000,323263000,0,정부내부수입및기타
212,213,2024,과학기술정보통신부,우편사업특별회계,자본계정,세계잉여금,세계잉여금이입액,전년도세계잉여금,보전수입,18965000,18965000,0,정부내부수입및기타


In [43]:
# 과목분류별 count
print(세입예산편성_DF.세입과목분류.value_counts())

# 세입과목분류별 국회확정금액 집계 및 억단위 변환
세입유형별세입 = pd.DataFrame(세입예산편성_DF.groupby('세입과목분류')['국회확정금액'].sum()/100000)
세입유형별세입.국회확정금액 = round(세입유형별세입.국회확정금액)
print(세입유형별세입)

#출력순서 지정
print(세입유형별세입.iloc[[0,2,3,1],:])

세입과목분류
자체수입         1309
정부내부수입및기타     423
국세수입           19
여유자금회수          2
Name: count, dtype: int64
           국회확정금액
세입과목분류           
국세수입      3673140
여유자금회수     218848
자체수입      2599724
정부내부수입및기타 9241362
           국회확정금액
세입과목분류           
국세수입      3673140
자체수입      2599724
정부내부수입및기타 9241362
여유자금회수     218848
